In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import glob
from google.colab import files
from google.colab import drive
drive.mount('/gdrive', force_remount=True)



Mounted at /gdrive


# 음성데이터 세션 나누기

In [4]:
AI_root = glob.glob(r"/gdrive/MyDrive/빅데이터 45기_파이널 프로젝트/음원데이터/AI/*")
ori_root = glob.glob(r"/gdrive/MyDrive/빅데이터 45기_파이널 프로젝트/음원데이터/원본/*")
X_path = AI_root + ori_root
print(X_path)

['/gdrive/MyDrive/빅데이터 45기_파이널 프로젝트/음원데이터/AI/AI.vampire - ROSÉ (ai cover).wav', "/gdrive/MyDrive/빅데이터 45기_파이널 프로젝트/음원데이터/AI/AI.Taylor Swift IA - we can't be friends (wait for your love) (artificial intelligence version).wav", '/gdrive/MyDrive/빅데이터 45기_파이널 프로젝트/음원데이터/AI/AI.Passenger - LET HER GO (ED SHEERAN - A.I COVER LYRICS).wav', '/gdrive/MyDrive/빅데이터 45기_파이널 프로젝트/음원데이터/AI/AI.Jungkook - Off My Face (AI Cover).wav', '/gdrive/MyDrive/빅데이터 45기_파이널 프로젝트/음원데이터/AI/AI.Copy of Standing Next to You - AI theWeeknd.wav', '/gdrive/MyDrive/빅데이터 45기_파이널 프로젝트/음원데이터/AI/AI.Copy of Jungkook - Die For You (AI cover).wav', '/gdrive/MyDrive/빅데이터 45기_파이널 프로젝트/음원데이터/AI/AI.CHIQUITA Ai Cover “ 2002” (Anne Marie).wav', '/gdrive/MyDrive/빅데이터 45기_파이널 프로젝트/음원데이터/AI/AI.Avril Lavigne - August (Taylor Swift Cover).wav', '/gdrive/MyDrive/빅데이터 45기_파이널 프로젝트/음원데이터/AI/AI.Avril Lavigne - 3am (Halsey AI Cover).wav', '/gdrive/MyDrive/빅데이터 45기_파이널 프로젝트/음원데이터/원본/ori.Taylor Swift – august.wav', '/gdrive/MyDrive/빅데이터 45기_파이널 

In [5]:
pip install pydub

In [6]:
#음원 구간별 추출
from pydub import AudioSegment
import os

for a in X_path:
  # wav 파일 로드
  audio = AudioSegment.from_wav(a)

  # 분할 간격 설정 (밀리초 단위)
  interval = 10 * 1000  # 10초 = 10,000 밀리초
  step = 5 * 1000  # 5초 = 5,000 밀리초

  # 전체 길이
  length = len(audio) #밀리초로 환산됨


  # 저장할 디렉토리 설정
  output_dir = r'/gdrive/MyDrive/빅데이터 45기_파이널 프로젝트/음원데이터/save'

  # 분할 및 저장
  for i in range(0, length, step):
      start_time = i  # 시작 시간 (밀리초 단위)
      end_time = i + interval  # 종료 시간 (밀리초 단위)
      split_audio = audio[start_time:end_time]
      start_sec = start_time // 1000  # 시작 시간 (초 단위)
      end_sec = end_time // 1000  # 종료 시간 (초 단위)
      filename = os.path.join(output_dir, f"output_{start_sec}-{end_sec}_{os.path.basename(a)}.wav")
      split_audio.export(filename, format="wav")
      print(f"Saved {filename}")

  print("분할 완료")


Saved /gdrive/MyDrive/빅데이터 45기_파이널 프로젝트/음원데이터/save/output_0-10_AI.vampire - ROSÉ (ai cover).wav.wav
Saved /gdrive/MyDrive/빅데이터 45기_파이널 프로젝트/음원데이터/save/output_5-15_AI.vampire - ROSÉ (ai cover).wav.wav
Saved /gdrive/MyDrive/빅데이터 45기_파이널 프로젝트/음원데이터/save/output_10-20_AI.vampire - ROSÉ (ai cover).wav.wav
Saved /gdrive/MyDrive/빅데이터 45기_파이널 프로젝트/음원데이터/save/output_15-25_AI.vampire - ROSÉ (ai cover).wav.wav
Saved /gdrive/MyDrive/빅데이터 45기_파이널 프로젝트/음원데이터/save/output_20-30_AI.vampire - ROSÉ (ai cover).wav.wav
Saved /gdrive/MyDrive/빅데이터 45기_파이널 프로젝트/음원데이터/save/output_25-35_AI.vampire - ROSÉ (ai cover).wav.wav
Saved /gdrive/MyDrive/빅데이터 45기_파이널 프로젝트/음원데이터/save/output_30-40_AI.vampire - ROSÉ (ai cover).wav.wav
Saved /gdrive/MyDrive/빅데이터 45기_파이널 프로젝트/음원데이터/save/output_35-45_AI.vampire - ROSÉ (ai cover).wav.wav
Saved /gdrive/MyDrive/빅데이터 45기_파이널 프로젝트/음원데이터/save/output_40-50_AI.vampire - ROSÉ (ai cover).wav.wav
Saved /gdrive/MyDrive/빅데이터 45기_파이널 프로젝트/음원데이터/save/output_45-55_AI.vampire - ROSÉ (

# 모델링 시작

In [7]:
#구간별 추출한 데이터 경로 저장
X_path_new = glob.glob(r"/gdrive/MyDrive/빅데이터 45기_파이널 프로젝트/음원데이터/save/*")
print(len(X_path_new))

800


In [9]:
type(X_path_new)

list

In [10]:
import ntpath # 특정 경로에서 파일들을 가져오는 라이브러리
y = np.empty((0, 1)) # 비어있는 리스트 만들기

for f in X_path_new:
    if 'ori' in ntpath.basename(f): #  음성 데이터가 있는 디렉토리의 데이터가 '원본' 음성 : 0
        resp = np.array([0])  #   [0]
    elif 'AI' in ntpath.basename(f): # 음성 데이터가 있는 디렉토리의 데이터가 'AI' 음성 : 1
        resp = np.array([1])  # [1]
    resp = resp.reshape(1, 1)
    y = np.vstack((y, resp))
print (y)

[[1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.

In [11]:
# 데이터셋 나누기
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_path_new, y, test_size=0.25, random_state=42)
print(len(X_train)) # 600
print(len(X_test)) # 200

600
200


In [12]:
# 첫 번째 파일의 샘플 레이트 확인
sample_rate = librosa.load(X_train[0],sr = 16000)[1]
print(sample_rate)

16000


In [18]:
def preprocess_audio(audio_file, sample_rate=16000, n_fft=400, hop_length=160):
    # Load audio file
    y, sr = librosa.load(audio_file, sr=sample_rate)

    # STFT
    stft = librosa.stft(y, n_fft=n_fft, hop_length=hop_length)
    magnitude, phase = librosa.magphase(stft)

    # Log-amplitude spectrogram
    log_spec = librosa.amplitude_to_db(magnitude)

    return log_spec

In [19]:
def load_and_process_data(file_list):
    processed_data = []
    for file in file_list:
        processed_audio = preprocess_audio(file)
        processed_data.append(processed_audio)
    return processed_data

In [20]:
X_train2 = load_and_process_data(X_train)
X_test2 = load_and_process_data(X_test)

/usr/local/lib/python3.10/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=400 is too large for input signal of length=272
  warnings.warn(


In [29]:
X_train2

[array([[  3.8110714,   3.421876 ,   2.2621658, ...,   2.127627 ,
          -2.9617233,   9.791063 ],
        [  8.039296 ,  10.981383 ,  17.289434 , ...,  23.994411 ,
          24.685589 ,  20.812527 ],
        [ 12.558361 ,  17.018457 ,  13.3729   , ...,  27.369164 ,
          25.93646  ,  23.983227 ],
        ...,
        [-30.140781 , -40.268303 , -40.268303 , ..., -40.268303 ,
         -27.212927 ,  -6.874387 ],
        [-30.36725  , -40.268303 , -40.268303 , ..., -40.268303 ,
         -27.53895  ,  -6.9183264],
        [-30.412977 , -40.268303 , -40.268303 , ..., -40.268303 ,
         -27.552721 ,  -6.9510245]], dtype=float32),
 array([[ 18.868095 ,  -3.3056962,   7.276499 , ...,  24.145988 ,
          29.39231  ,  12.29102  ],
        [ 14.9669695,  18.968594 ,  19.971197 , ...,  34.548557 ,
          33.11943  ,  24.023943 ],
        [  8.261793 ,  24.579052 ,  18.114658 , ...,  36.076935 ,
          30.446833 ,  25.681541 ],
        ...,
        [-24.182455 , -42.10311  , -42.

In [28]:
print("Train data shape:", X_train2.shape)
print("Test data shape:", X_test2.shape)

AttributeError: 'list' object has no attribute 'shape'

In [27]:
from keras import layers
from keras import models
from keras import optimizers
from keras import losses
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout


In [31]:
# 신경망 구축
model = Sequential()
model.add(Conv1D(64, kernel_size=3, activation='relu', input_shape=(101,1)))
model.add(MaxPooling1D(pool_size=2))
model.add(Conv1D(128, kernel_size=3, activation='relu'))
model.add(MaxPooling1D(pool_size=2))
model.add(Conv1D(256, kernel_size=3, activation='relu'))
model.add(MaxPooling1D(pool_size=2))
model.add(Flatten())
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv1d_3 (Conv1D)           (None, 99, 64)            256       
                                                                 
 max_pooling1d_3 (MaxPoolin  (None, 49, 64)            0         
 g1D)                                                            
                                                                 
 conv1d_4 (Conv1D)           (None, 47, 128)           24704     
                                                                 
 max_pooling1d_4 (MaxPoolin  (None, 23, 128)           0         
 g1D)                                                            
                                                                 
 conv1d_5 (Conv1D)           (None, 21, 256)           98560     
                                                                 
 max_pooling1d_5 (MaxPoolin  (None, 10, 256)          

In [32]:
# 학습환경
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [33]:
history = model.fit(X_train2, y_train, epochs=50, batch_size=32, validation_data=(X_test2,y_test))
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.legend(['acc','val_acc'])
plt.title('Model ACC')
plt.xlabel('Epochs')
plt.ylabel('ACC')

ValueError: Data cardinality is ambiguous:
  x sizes: 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 201, 2